# Donor Models — Inference

Loads both donor models and writes predictions to `operational.donor_predictions`.

| Column | Description |
|---|---|
| `predicted_churn` | True / False |
| `prob_churn` | Probability of churning |
| `predicted_don_type` | Most likely next donation type |
| `prob_monetary` – `prob_skills` | Per-type probabilities |

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd

from config import (
    DONOR_CHURN_METADATA_PATH, DONOR_CHURN_MODEL_PATH,
    DONOR_TYPE_METADATA_PATH, DONOR_TYPE_MODEL_PATH,
    OPERATIONAL_SCHEMA,
)
from utils_db import ensure_donor_predictions_table, get_engine, pg_conn

print('Imports loaded.')

## Load models

In [ ]:
churn_model = joblib.load(str(DONOR_CHURN_MODEL_PATH))
type_model  = joblib.load(str(DONOR_TYPE_MODEL_PATH))

with open(DONOR_CHURN_METADATA_PATH) as f:
    churn_meta = json.load(f)
with open(DONOR_TYPE_METADATA_PATH) as f:
    type_meta = json.load(f)

print(f"Churn model v{churn_meta['model_version']}")
print(f"Type  model v{type_meta['model_version']} | classes: {type_meta['classes']}")

## Build features

In [ ]:
from inference_donor import _build_features

engine = get_engine(OPERATIONAL_SCHEMA)
df = _build_features(engine)
print(f'Scoring {len(df)} supporters')
df.head()

## Run predictions

In [ ]:
churn_features = churn_meta['feature_cols']
type_features  = type_meta['feature_cols']
type_classes   = type_meta['classes']

X_churn = df[churn_features]
X_type  = df[type_features]

churn_probs = churn_model.predict_proba(X_churn)
churn_preds = churn_model.predict(X_churn)
type_probs  = type_model.predict_proba(X_type)
type_preds  = type_model.predict(X_type)

type_prob_df   = pd.DataFrame(type_probs, columns=type_classes)
churn_idx      = list(churn_model.classes_).index(1) if 1 in list(churn_model.classes_) else -1
ts             = datetime.now(timezone.utc).isoformat()

rows = []
for i, (_, row) in enumerate(df.iterrows()):
    rows.append((
        int(row['supporter_id']),
        bool(churn_preds[i]),
        float(churn_probs[i][churn_idx]) if churn_idx >= 0 else None,
        str(type_preds[i]),
        float(type_prob_df.iloc[i].get('Monetary', 0)),
        float(type_prob_df.iloc[i].get('In-Kind', 0)),
        float(type_prob_df.iloc[i].get('Time', 0)),
        float(type_prob_df.iloc[i].get('Skills', 0)),
        ts,
    ))

print(f'Built {len(rows)} prediction rows')

## Write to database

In [ ]:
ensure_donor_predictions_table(OPERATIONAL_SCHEMA)

with pg_conn(OPERATIONAL_SCHEMA) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO operational.donor_predictions
                (supporter_id, predicted_churn, prob_churn, predicted_don_type,
                 prob_monetary, prob_in_kind, prob_time, prob_skills, prediction_ts)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (supporter_id) DO UPDATE SET
                predicted_churn    = EXCLUDED.predicted_churn,
                prob_churn         = EXCLUDED.prob_churn,
                predicted_don_type = EXCLUDED.predicted_don_type,
                prob_monetary      = EXCLUDED.prob_monetary,
                prob_in_kind       = EXCLUDED.prob_in_kind,
                prob_time          = EXCLUDED.prob_time,
                prob_skills        = EXCLUDED.prob_skills,
                prediction_ts      = EXCLUDED.prediction_ts
        """, rows)

print(f'Wrote {len(rows)} predictions to donor_predictions')
print(f'Timestamp: {ts}')